# Application Prioritization Scorecard
This notebook computes risk and impact scores for each application using the enriched inventory dataset.

In [ ]:
import pandas as pd
from pathlib import Path
DATASET = Path('../..') / 'datasets' / 'landing' / 'windows2016_inventory.parquet'
inventory = pd.read_parquet(DATASET)
inventory[['application', 'criticality', 'rto_hours', 'rpo_hours']].head()

In [ ]:
CRITICALITY_MAP = {'High': 3, 'Medium': 2, 'Low': 1}
def compute_scores(row):
    criticality_score = CRITICALITY_MAP.get(row.get('criticality'), 1)
    rto = row.get('rto_hours') or 24
    rpo = row.get('rpo_hours') or 24
    risk = criticality_score * (24 / max(rto, 1))
    impact = criticality_score * (24 / max(rpo, 1))
    return pd.Series({'risk_score': risk, 'impact_score': impact})
scores = inventory.apply(compute_scores, axis=1)
inventory = pd.concat([inventory, scores], axis=1)
inventory[['application', 'risk_score', 'impact_score']].sort_values('risk_score', ascending=False)

In [ ]:
OUTPUT = Path('../..') / 'datasets' / 'landing' / 'application_scorecards.parquet'
inventory.to_parquet(OUTPUT, index=False)
OUTPUT